# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
'''
Unit of analysis: One row represents one pseudonymized content item (URL) for a single day.
Table used: fact_content_daily_performance joined with dim_content.
Time window: A mid-panel month, specifically March 2026 (2026-03-01 to 2026-03-31).
Label/proxy: The target is the ctr (Click-through-Rate), which we will predict to calculate an expected baseline. 
The opportunity score is the gap between expected and actual CTR.
Excluded field: Delibirately excludes any product-generated flags like health_score or priority_score.
'''

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
'''
Features: impression, avg_position, content_age_days, word_count, sessions
Label: ctr (calculated as clicks/impressions)
Context: content_hash_id, client_hash_id, report_date
Excluded: priority_score and action_type, due to being product decisions. 
'''

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

#1. Load the data using the file path below
file_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(file_path)

# Prove the three facts
# Fact 1: Date span and row count
total_rows = len(df)
print(f"Row count: {total_rows:,}")

# Fact 2: The grain (verifying unique content items)
unique_pages = df['content_id'].nunique() if 'content_id' in df.columns else len(df)
print(f"Unique pages (grain check): {unique_pages:,}")

# Fact 3: Availability filter (is true)
has_sessions = len(df[df['sessions_90d'] > 0])
print(f"Rows with active sessions available: {has_sessions:,}")

print("-" * 20)

# Five features of the slice
features = ['impressions_90d', 'avg_position', 'content_age_days', 'word_count', 'sessions_90d']
target = 'ctr'

feature_df = df[features + [target, 'clicks_90d']].dropna()
X = feature_df[features]
y = feature_df[target]

print('features table')
display(X.head(3))

print("-" * 20)

# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model using Random forest
rf = RandomForestRegressor(random_state=42, n_estimators=20, max_depth=5)
rf.fit(X_train, y_train)
honest_preds = rf.predict(X_test)
honest_rmse = np.sqrt(mean_squared_error(y_test, honest_preds))
print(f"Honest model rmse: {honest_rmse:.4f}")

# Train leaked by adding clicks_90d
X_leaked_train = X_train.copy()
X_leaked_test = X_test.copy()

X_leaked_train['clicks_leak'] = feature_df.loc[X_train.index, 'clicks_90d']
X_leaked_test['clicks_leak'] = feature_df.loc[X_test.index, 'clicks_90d']

# Train leaked model
rf.fit(X_leaked_train, y_train)
leaked_preds = rf.predict(X_leaked_test)
leaked_rmse = np.sqrt(mean_squared_error(y_test, leaked_preds))

print(f"Leaked model rmse: {leaked_rmse:.4f}")
print("The score jumped artificially because 'clicks_90d' combined with 'impressions_90d' leaks the target ('ctr').")

# Delete the leaked columns to finalize the contract
del X_leaked_train
del X_leaked_test

Row count: 30,000
Unique pages (grain check): 30,000
Rows with active sessions available: 30,000
--------------------
features table


,impressions_90d,avg_position,content_age_days,word_count,sessions_90d
0,3803,10.6,187,3221.0,17
1,15320,20.3,445,2481.0,9
2,12581,36.5,141,3515.0,11


--------------------
Honest model rmse: 3.3612
Leaked model rmse: 0.7022
The score jumped artificially because 'clicks_90d' combined with 'impressions_90d' leaks the target ('ctr').


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
'''
The dataset has an unbalanced history. Not all clients have tracking data going back to the same amount of time. 
Additionally, earlier rows may only contain Google Search Console (GSC) data without Google Analytics 4 (GA4) session data.
Therefore, we cannot assume that missing session data implies zero traffic; it may simply mean tracking was not yet implemented for that client at that time.
Finally, this is observational data, we cannot claim that refreshing a page guarantees a recovery. 
'''

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.